# Checkpoint-chaining probe

Tests the one mechanism the whole multi-session training plan rests on:
a private dataset written by one session must be **readable by the next**.

Motivation: `kaggle datasets files` and `datasets download` both return 403
on this account's own private dataset from outside Kaggle, even though
`datasets list --mine` sees it and `datasets version` writes to it happily.
That 403 is an API-surface permission. Inside a kernel the dataset is
**mounted**, not fetched over the API — so it may be entirely unaffected.
This notebook settles which.

If the mount works, checkpoint chaining is viable and the 403 is a local
inconvenience. If it does not, the multi-session plan needs redesigning
before a single GPU-hour is spent.

In [ ]:
# 1. did the private dataset mount at all?
from pathlib import Path

INPUT = Path("/kaggle/input")
print("attached:", [p.name for p in INPUT.iterdir()] if INPUT.exists() else "NONE")
for p in sorted(INPUT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(INPUT), f"({p.stat().st_size}B)")

In [ ]:
# 2. is the checkpoint readable? This is the question.
cands = sorted(INPUT.rglob("ckpt_step*.pt"))
print("checkpoints found:", [str(c.relative_to(INPUT)) for c in cands])
assert cands, "PRIVATE DATASET DID NOT MOUNT -- chaining is not viable as designed"

content = cands[0].read_bytes()
print("read", len(content), "bytes ->", content[:60])
print("PRIVATE DATASET MOUNTS AND READS -- chaining viable")

In [ ]:
# 3. does our own latest_checkpoint() pick it up? The real chaining entry point,
#    not a hand-rolled glob.
import subprocess, sys

r = subprocess.run(
    ["git", "clone", "-q", "--branch", "feat/m0-m1-harness", "--depth", "1",
     "https://github.com/SonLamHG/pgmm.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
assert r.returncode == 0, r.stderr
sys.path.insert(0, "/kaggle/working/repo")

from kaggle_harness.chain import latest_checkpoint

found = latest_checkpoint([cands[0].parent, Path("/kaggle/working/ckpt")])
print("latest_checkpoint ->", found)
assert found is not None, "latest_checkpoint failed to see the mounted checkpoint"
print("CHAIN ENTRY POINT OK")

In [ ]:
# 4. THE DECISIVE TEST (D16): can a kernel write a dataset version back?
#
#    Two independent unknowns, both fatal if wrong, both settled here:
#      a) does a UI-attached Secret survive an API-driven kernel push?
#         (the `kaggle` package has no secret support at all, so the attachment
#          is UI-only while our whole workflow is API-driven)
#      b) does a *legacy* key authorise `datasets version` from in here?
#         (legacy keys do dataset ops but not kernel ops -- see D15)
import os
from pathlib import Path

secret = None
try:
    from kaggle_secrets import UserSecretsClient

    secret = UserSecretsClient().get_secret("KAGGLE_KEY")
    print("(a) SECRET READABLE after an API push -- len:", len(secret))
except Exception as e:
    print("(a) SECRET NOT READABLE:", type(e).__name__, e)
    print("    -> either the toggle is off for this notebook, or UI attachment")
    print("       does not survive an API push. Fall back to a private dataset")
    print("       holding the key, or to local orchestration (D16).")

assert secret, "no secret -> cannot test (b)"

In [ ]:
# 5. (b) close the loop for real: write a checkpoint from THIS kernel and push
#    it as a new dataset version, exactly as a dying training session would.
import json
import subprocess

# the CLI reads these from the environment; username is not a secret
os.environ["KAGGLE_USERNAME"] = "snlmhong"
os.environ["KAGGLE_KEY"] = secret

STAGE = Path("/kaggle/working/ckpt_push")
STAGE.mkdir(parents=True, exist_ok=True)

# carry the existing checkpoint forward, then add a new one -- a dataset version
# replaces the whole contents, so anything omitted here is deleted upstream.
for old in sorted(INPUT.rglob("ckpt_step*.pt")):
    (STAGE / old.name).write_bytes(old.read_bytes())

marker = "written from inside kernel pgmm-chainprobe\n"
(STAGE / "ckpt_step000099.pt").write_text(marker)
(STAGE / "dataset-metadata.json").write_text(json.dumps(
    {"title": "pgmm-ckpt", "id": "snlmhong/pgmm-ckpt",
     "licenses": [{"name": "CC0-1.0"}]}, indent=2))

print("staging:", sorted(p.name for p in STAGE.iterdir()))

r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", str(STAGE),
     "-m", "chainprobe: written from inside a kernel", "--dir-mode", "zip"],
    capture_output=True, text=True,
)
print("exit:", r.returncode)
print((r.stdout or "")[-1500:])
print((r.stderr or "")[-1500:])

if r.returncode == 0:
    print("\n(b) LEGACY KEY AUTHORISES datasets version FROM A KERNEL")
    print("D16 RESOLVED -- checkpoint chaining works end to end")
else:
    print("\n(b) FAILED -- legacy key insufficient from a kernel.")
    print("Fall back to local orchestration (D16).")